# Data Cleaning and Preparation
## DAT540 Introduction to Data Science
## University of Stavanger
### L09

In [39]:
import numpy as np
import pandas as pd

- Most real world data sets have missing values
- Such values could lead to erroneous results
- Proper strategies should be applied prior to any data processing in order to handle missing data

- **Handling Missing Data**
- pandas uses floating point value *NaN* (Not a Number) to represent missing values
- Such values are termed as *sentinel values* and pandas provides a wide array of methods to detect them
- pandas also employees a sentinel value *NA* (Not Available) to represent data that does not exist
- When cleaning data for analysis, it is crucial to perform a prior analysis on missing data to identify data collection issues or detect biases
- The built in Python *None* value is also treated as a NA object
- Typically, if we have a Series or DataFrame, we can use the instance method *isnull* to detect missing values
- *isnull* instance method returns true for all values that are  null
- Other instance methods such as
  - *dropna* filters axis labels based on whether values for each label have missing data, with varying thresholds for how much missing data to tolerate
  - *fillna* fills in missing data with some value or using interpolation methods such as 'ffill'or 'bfill'  

In [40]:
titanic = pd.read_csv('/content/titanicv1.csv', delimiter=',')
print(titanic.shape)
titanic.head(20)


(156, 13)


,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,7,8,0,3,"Palsson, Master. Gosta Leonard",NaN,2.0,3,1,349909,21.0750,NaN,S
8,8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",NaN,14.0,1,0,237736,30.0708,NaN,C


In [41]:
titanic.isnull().head()

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,False,False,False,False,False,False,False,False,False,False,False,True,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,False,True,False


- How do we get an overview of the missing values per column

In [42]:
titanic.isnull().sum(axis=0)

,0
Unnamed: 0,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,50
Age,30
SibSp,0
Parch,0
Ticket,0


- rows or columns with missing values can be dropped with the instance method *dropna*
- *axis* argument determines either a row or column to be dropped
- *how* {'any', 'all'}, default 'any'. determine when we remove, do we have at least one NA or all NA.
- *thresh* require that many non-NA values to be present
- subset : array-like, labels along an axis to consider, e.g. if you are dropping rows these would be a list of columns to include.


import pandas as pd

df = pd.DataFrame({
    "Name": ["John", "Mary", "Alex"],
    "Age": [22, None, 30],
    "Cabin": ["A12", "B20", None]
})


df.dropna(subset=["Age"])

Pandas asks:
"Is Age missing?"

- John → Age = 22 → keep
- Mary → Age = NaN → drop
- Alex → Age = 30 → keep

In [43]:

df = pd.DataFrame({
    "Name": ["John", "Mary", "Alex"],
    "Age": [22, None, 30],
    "Cabin": ["A12", "B20", None]
})
df

,Name,Age,Cabin
0,John,22.0,A12
1,Mary,NaN,B20
2,Alex,30.0,None


In [44]:
df.dropna(subset=["Age", "Cabin"])

,Name,Age,Cabin
0,John,22.0,A12


In [45]:
#Only look at these particular columns when deciding whether a row should be dropped.

titanic.dropna(subset=['Embarked'], inplace=True)

In [46]:
titanic.shape

(155, 13)

In [47]:
titanic.isnull().sum(axis=0)

,0
Unnamed: 0,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,50
Age,30
SibSp,0
Parch,0
Ticket,0


- What about the Cabin column it has around 80% missing data
- Do we drop this column? May be we can interpolate the values by looking at the Fare column

In [48]:
# We drop the cabin column as it has a lot of missing data and if we can interpolate it from the Fare or Ticket column,
# then we dont need it
# We provide arg thresh=100, as we need a column to have atleast 60% non missing values
#Keep an axis only if it has at least 100 non-missing values.
titanic.dropna(thresh=100, axis=1, inplace=True)

In [49]:
titanic.shape

(155, 12)

- We also want to drop all rows that have all NaN values

In [50]:
titanic.dropna(how='all', inplace=True)
print(titanic.shape)
titanic.isnull().sum(axis=0)

(155, 12)


,0
Unnamed: 0,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,50
Age,30
SibSp,0
Parch,0
Ticket,0


- Filling in Missing Data
- The *fillna* instance method is the workhorse for filling holes in a dataset
- It's *value* argument takes in a *scalar*, *dict*, *Series* or *DataFrame*. The dict/Series/DataFrame of values specify which values to use for each index (Series) or column (DataFrame)
- *method* argument {'bfill', 'pad', 'ffill', None} (defualt None) specifies the method to use to fill holes
- *limit* argument could be provided in tandem with the method argument. It specifies the maximum number of consecutive NaN values to forward/backward fill. It thus could create partially filled row or column

- For the titanic dataset, let us try to fill the age column
- What would be good strategy?

In [51]:
titanic.head()

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


In [52]:
titanic.loc[:, ['Age', 'Fare']]

,Age,Fare
0,22.0,7.2500
1,38.0,71.2833
2,26.0,7.9250
3,35.0,53.1000
4,35.0,8.0500
...,...,...
151,22.0,66.6000
152,55.5,8.0500
153,40.5,14.5000
154,NaN,7.3125


In [53]:
age_fare = titanic.dropna(subset=['Age']).loc[:, ['Age', 'Fare']]
age_fare

,Age,Fare
0,22.0,7.2500
1,38.0,71.2833
2,26.0,7.9250
3,35.0,53.1000
4,35.0,8.0500
...,...,...
150,51.0,12.5250
151,22.0,66.6000
152,55.5,8.0500
153,40.5,14.5000


- May be the Fare for children are cheaper?

In [54]:
# First let us check if there is any correlation between the non missing Age and Fare values
#First remove passengers with missing Age, then keep all remaining passengers but only the Age and Fare columns.

age_fare = titanic.dropna(subset=['Age']).loc[:, ['Age', 'Fare']]
print('shape: ', age_fare.shape)
age_fare['Age'].corr(age_fare['Fare']) # We see almost no correlation

#Age and Fare have very little linear correlation.

shape:  (125, 2)


np.float64(0.018210834187746498)

In [55]:
titanic['Survived'].corr(titanic['Fare'])

np.float64(0.017836717815619803)

- Is there any relationship, if we create bins for Age and Fare
- Ex.
  - Age: 0 -> [0-6], 1 -> [6-12], 2 -> [12-18], 3-> [18+]
  - Fare: round the fare to the nearest integer


In [56]:

#Apply this function to every value in the Age column.
age_fare['Age'].map(lambda x: 0 if x < 6 else 1 if x < 12 else 2 if x < 18 else 3).corr(age_fare['Fare'].apply(lambda x: np.ceil(x)))

np.float64(0.04135740503165592)

- From the first look Age and Fare does not have any strong correlations
- We don't directly see any other column that might have relevance to Age.
- Let us fill it with the mean value for now

In [57]:
titanic.fillna({'Age': titanic['Age'].mean()}, inplace=True)

In [59]:
titanic.isnull().sum(axis=0)

,0
Unnamed: 0,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,50
Age,0
SibSp,0
Parch,0
Ticket,0


- How can we fill Sex column?
- Is there any relation with any other column? Can we derive from it?

In [60]:
titanic.head(30)

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.00000,1,0,A/5 21171,7.2500,S
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00000,1,0,PC 17599,71.2833,C
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.00000,0,0,STON/O2. 3101282,7.9250,S
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00000,1,0,113803,53.1000,S
4,4,5,0,3,"Allen, Mr. William Henry",male,35.00000,0,0,373450,8.0500,S
5,5,6,0,3,"Moran, Mr. James",male,28.06264,0,0,330877,8.4583,Q
6,6,7,0,1,"McCarthy, Mr. Timothy J",male,54.00000,0,0,17463,51.8625,S
7,7,8,0,3,"Palsson, Master. Gosta Leonard",NaN,2.00000,3,1,349909,21.0750,S
8,8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.00000,0,2,347742,11.1333,S
9,9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",NaN,14.00000,1,0,237736,30.0708,C


In [69]:
title_to_sex = {
    'Mr': 'male',
    'Master': 'male',
     'Rev': 'male',
     'Don': 'male',
    'Mrs': 'female',
    'Miss': 'female',
}

In [70]:
titanic['Title'].value_counts()

,count
Title,
Mr,89
Miss,33
Mrs,22
Master,8
Rev,2
Don,1


In [68]:
titanic['Title'].loc[titanic['Title'] == 'Rev']

,Title
149,Rev
150,Rev


In [71]:
titanic['Title'] = titanic['Name'].str.extract(r',\s*([^.]*)\.')

In [72]:
titanic

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Title
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.00000,1,0,A/5 21171,7.2500,S,Mr
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00000,1,0,PC 17599,71.2833,C,Mrs
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.00000,0,0,STON/O2. 3101282,7.9250,S,Miss
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00000,1,0,113803,53.1000,S,Mrs
4,4,5,0,3,"Allen, Mr. William Henry",male,35.00000,0,0,373450,8.0500,S,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,151,152,1,1,"Pears, Mrs. Thomas (Edith Wearne)",female,22.00000,1,0,113776,66.6000,S,Mrs
152,152,153,0,3,"Meo, Mr. Alfonzo",male,55.50000,0,0,A.5. 11206,8.0500,S,Mr
153,153,154,0,3,"van Billiard, Mr. Austin Blyler",male,40.50000,0,2,A/5. 851,14.5000,S,Mr
154,154,155,0,3,"Olsen, Mr. Ole Martin",male,28.06264,0,0,Fa 265302,7.3125,S,Mr


In [73]:
titanic['Sex'] = titanic['Sex'].fillna(titanic['Title'].map(title_to_sex))

In [74]:
titanic.head(30)

,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Title
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22.00000,1,0,A/5 21171,7.2500,S,Mr
1,1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.00000,1,0,PC 17599,71.2833,C,Mrs
2,2,3,1,3,"Heikkinen, Miss. Laina",female,26.00000,0,0,STON/O2. 3101282,7.9250,S,Miss
3,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.00000,1,0,113803,53.1000,S,Mrs
4,4,5,0,3,"Allen, Mr. William Henry",male,35.00000,0,0,373450,8.0500,S,Mr
5,5,6,0,3,"Moran, Mr. James",male,28.06264,0,0,330877,8.4583,Q,Mr
6,6,7,0,1,"McCarthy, Mr. Timothy J",male,54.00000,0,0,17463,51.8625,S,Mr
7,7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.00000,3,1,349909,21.0750,S,Master
8,8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.00000,0,2,347742,11.1333,S,Mrs
9,9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.00000,1,0,237736,30.0708,C,Mrs


In [75]:
titanic.isnull().sum(axis=0)

,0
Unnamed: 0,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0


In [ ]:
print(titanic.sample(3))

     Unnamed: 0  PassengerId  Survived  Pclass                          Name  \
145         145          146         0       2  Nicholls, Mr. Joseph Charles   
72           72           73         0       2          Hood, Mr. Ambrose Jr   
111         111          112         0       3          Zabour, Miss. Hileni   

        Sex   Age  SibSp  Parch        Ticket     Fare Embarked  
145    male  19.0      1      1    C.A. 33112  36.7500        S  
72     male  21.0      0      0  S.O.C. 14879  73.5000        S  
111  female  14.5      1      0          2665  14.4542        C  


In [79]:
from sklearn.linear_model import LogisticRegression

scikit-learn
     │
     └── linear_model
             │
             └── LogisticRegression
                       ↓
                 model = LogisticRegression()
                       ↓
                 model.fit(X_train, y_train)
                       ↓
                 model.predict(X_test)

In [77]:
#For the Titanic dataset, Logistic Regression can be used to predict:
#Survived = 0 → did not survive
#Survived = 1 → survived



tdataset = titanic[['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Survived']]


# We take 80% of the dataset to train and rest to test
# there are further optimization that can be performed on the data. This is just an initial example
test = tdataset.sample(frac=.2)
train = tdataset[~tdataset.isin(test).all(1)]

In [80]:
# Train the model

#This creates an empty Logistic Regression model. The model needs to learn from our training data.
model = LogisticRegression()


# First we test the training scores
_ = model.fit(train.loc[:, train.columns != 'Survived'], train['Survived'])

In [82]:
# Training error
predictions = pd.DataFrame([])
predictions['Survived'] = np.ceil(model.predict(train.loc[:, train.columns != 'Survived']))

predictions

,Survived
0,0
1,0
2,0
3,0
4,0
...,...
119,0
120,1
121,0
122,0


In [ ]:
#Convert predictions to integers
predictions['Survived'] = predictions['Survived'].map(lambda x: int(x))
predictions['Survived']

,Survived
0,0
1,0
2,0
3,0
4,0
...,...
119,0
120,0
121,0
122,0


In [83]:
#This creates a new column containing the index of each passenger.
predictions['PassengerId'] = train.index.values
predictions.set_index('PassengerId', inplace=True)
predictions.sample(5)

,Survived
PassengerId,
90,0
49,0
43,1
67,0
146,0


In [84]:
# since we are doing random sampling in choosing the train and test set, expect different MAPE across multiple runs
compare = train['Survived'] == predictions['Survived']
print('accuracy:', compare.sum() / len(compare) * 100)

accuracy: 70.96774193548387


In [85]:
# testing error
predictions = pd.DataFrame([])
predictions['Survived'] = np.ceil(model.predict(test.loc[:, test.columns != 'Survived']))
predictions['Survived'] = predictions['Survived'].map(lambda x: int(x))
predictions['PassengerId'] = test.index.values
predictions.set_index('PassengerId', inplace=True)
# since we are doing random sampling in choosing the train and test set, expect different MAPE across multiple runs
compare = test['Survived'] == predictions['Survived']
print('accuracy:', compare.sum() / len(compare) * 100)

accuracy: 67.74193548387096
